# Classical CART and Cost-Complexity Pruning

This notebook demonstrates the classical single-tree workflow: grow a CART classifier, inspect the learned rules, then prune the tree using the cost-complexity path.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

RANDOM_STATE = 7

In [ ]:
X, y = make_classification(
    n_samples=1500,
    n_features=8,
    n_informative=4,
    n_redundant=2,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.04,
    random_state=RANDOM_STATE,
)
feature_names = [f"x{i}" for i in range(X.shape[1])]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

In [ ]:
full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
full_tree.fit(X_train, y_train)

shallow_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=RANDOM_STATE)
shallow_tree.fit(X_train, y_train)

def score_model(name, model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    return {
        "model": name,
        "depth": model.get_depth(),
        "leaves": model.get_n_leaves(),
        "accuracy": accuracy_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    }

pd.DataFrame([
    score_model("unpruned CART", full_tree),
    score_model("depth-limited CART", shallow_tree),
])

In [ ]:
print(export_text(shallow_tree, feature_names=feature_names, decimals=2))

In [ ]:
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

rows = []
models = []
for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha)
    model.fit(X_train, y_train)
    rows.append(score_model(alpha, model))
    models.append(model)

pruning = pd.DataFrame(rows).rename(columns={"model": "ccp_alpha"})
pruning.head(), pruning.tail()

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(pruning["ccp_alpha"], pruning["roc_auc"], marker="o", ms=3, label="ROC AUC")
ax1.set_xscale("log")
ax1.set_xlabel("ccp_alpha")
ax1.set_ylabel("Test ROC AUC")
ax1.grid(True, alpha=0.25)
ax2 = ax1.twinx()
ax2.plot(pruning["ccp_alpha"], pruning["leaves"], color="tab:orange", marker="s", ms=3, label="Leaves")
ax2.set_ylabel("Leaves")
fig.suptitle("Cost-complexity pruning trades accuracy for simplicity")
fig.tight_layout()

In [ ]:
best_idx = pruning["roc_auc"].idxmax()
best_alpha = pruning.loc[best_idx, "ccp_alpha"]
pruned_tree = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha)
pruned_tree.fit(X_train, y_train)

pd.DataFrame([
    score_model("unpruned CART", full_tree),
    score_model("best pruned CART", pruned_tree),
    score_model("depth-limited CART", shallow_tree),
])

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(
    pruned_tree,
    feature_names=feature_names,
    class_names=["class 0", "class 1"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
)
plt.title("Top levels of the pruned tree")
plt.show()

## Takeaway

A single tree is readable, but it is high variance. Pruning makes the model smaller and often improves test behavior, but the best predictive model on tabular data is usually an ensemble.